In [ ]:
# -- coding: utf-8 --
# ===================================================================
#
#  Robust ADHD & Sex Prediction with FLAML
#  ---------------------------------------
#  Author: Proficient Data Scientist
#  Description: This script builds a two-stage prediction model.
#               1. Predicts sex using a robust XGBoost 'dart' model.
#               2. Uses the sex prediction as a meta-feature to predict
#                  ADHD outcome.
#               Hyperparameters for the ADHD model are tuned using
#               FLAML for efficiency and robustness.
#
# ===================================================================

# Imports
import pandas as pd
import numpy as np
import re
import warnings
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, f1_score, roc_curve
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import VarianceThreshold, SelectKBest, mutual_info_classif
from sklearn.impute import KNNImputer

from xgboost import XGBClassifier
from flaml import AutoML

warnings.filterwarnings("ignore")

# ===================================================================
# 1. GLOBAL CONFIGURATION
# ===================================================================
# --- Set the base directory for your data ---
DATA_DIR = Path(r"D:\eda") 

# --- File Paths ---
TRAIN_DIR = DATA_DIR / "TRAIN_NEW"
TEST_DIR = DATA_DIR / "TEST"

TRAIN_CAT_PATH = TRAIN_DIR / "TRAIN_CATEGORICAL_METADATA_new.xlsx"
TRAIN_CONN_PATH = TRAIN_DIR / "TRAIN_FUNCTIONAL_CONNECTOME_MATRICES_new_36P_Pearson.csv"
TRAIN_NUM_PATH = TRAIN_DIR / "TRAIN_QUANTITATIVE_METADATA_new.xlsx"
LABELS_PATH = TRAIN_DIR / "TRAINING_SOLUTIONS.xlsx"

TEST_CAT_PATH = TEST_DIR / "TEST_CATEGORICAL.xlsx"
TEST_CONN_PATH = TEST_DIR / "TEST_FUNCTIONAL_CONNECTOME_MATRICES.csv"
TEST_NUM_PATH = TEST_DIR / "TEST_QUANTITATIVE_METADATA.xlsx"

# --- Model & CV Parameters ---
SEED = 42
N_SPLITS = 10  # Increased for more robust CV
FLAML_TIME_BUDGET = 600  # Time in seconds for FLAML to search for params

np.random.seed(SEED)

# ===================================================================
# 2. UTILITY FUNCTIONS
# ===================================================================
def ensure_participant_id_column(df: pd.DataFrame) -> pd.DataFrame:
    """Standardizes the participant ID column name."""
    possible_ids = [col for col in df.columns if 'id' in col.lower()]
    if 'participant_id' in df.columns:
        return df
    elif possible_ids:
        return df.rename(columns={possible_ids[0]: 'participant_id'})
    else:
        raise ValueError("No participant ID column found!")

def youden_threshold(y_true, y_prob):
    """Calculates the optimal threshold using Youden's J statistic."""
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    j_scores = tpr - fpr
    best_thr = thresholds[np.argmax(j_scores)]
    return float(best_thr)

def f1_threshold(y_true, y_prob):
    """Calculates the optimal threshold for F1 score."""
    thr_list = np.linspace(0.05, 0.95, 50)
    best_thr, best_f1 = 0.5, -1.0
    for thr in thr_list:
        f1 = f1_score(y_true, (y_prob >= thr).astype(int))
        if f1 > best_f1:
            best_f1, best_thr = f1, thr
    return float(best_thr)

def compute_class_weight(y):
    """Computes scale_pos_weight for imbalanced classes."""
    pos = y.sum()
    neg = len(y) - pos
    return (neg + 1e-12) / (pos + 1e-12)

def kfold_oof_xgb(X, y, X_test, params, n_splits=N_SPLITS, early_stopping_rounds=50):
    """
    Performs stratified K-Fold cross-validation for XGBoost.
    Returns out-of-fold predictions, test predictions, and average best iteration.
    """
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    oof = np.zeros(len(X), dtype=float)
    test_preds = np.zeros(len(X_test), dtype=float)
    
    print(f"Starting {n_splits}-fold CV...")
    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), 1):
        X_tr, X_va = X[tr_idx], X[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]

        model = XGBClassifier(**params)
        
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_va, y_va)],
            eval_metric="auc",
            early_stopping_rounds=early_stopping_rounds,
            verbose=False
        )

        oof[va_idx] = model.predict_proba(X_va)[:, 1]
        test_preds += model.predict_proba(X_test)[:, 1] / n_splits
        
        best_iter = model.best_iteration if hasattr(model, "best_iteration") else params.get("n_estimators", 0)
        print(f"  [Fold {fold}] AUC: {roc_auc_score(y_va, oof[va_idx]):.4f}, Best Iter: {best_iter}")

    overall_auc = roc_auc_score(y, oof)
    print(f"-> Overall OOF AUC: {overall_auc:.4f}\n")
    return oof, test_preds

# ===================================================================
# 3. DATA LOADING AND PREPROCESSING
# ===================================================================
print("Loading data...")
train_cat = ensure_participant_id_column(pd.read_excel(TRAIN_CAT_PATH))
train_conn = pd.read_csv(TRAIN_CONN_PATH)
train_num = pd.read_excel(TRAIN_NUM_PATH)
labels = pd.read_excel(LABELS_PATH)

test_cat = ensure_participant_id_column(pd.read_excel(TEST_CAT_PATH))
test_conn = pd.read_csv(TEST_CONN_PATH)
test_num = pd.read_excel(TEST_NUM_PATH)

# Align labels with training data
labels = labels.set_index('participant_id').loc[train_cat['participant_id'].values].reset_index()

# Merge datasets
combined_cat = pd.concat([train_cat, test_cat], axis=0, ignore_index=True)
combined_num = pd.concat([train_num, test_num], axis=0, ignore_index=True)
combined_conn = pd.concat([train_conn, test_conn], axis=0, ignore_index=True)

# Process connectome features
print("Processing connectome features...")
pattern = r'(\d+)throw_(\d+)thcolumn'
rows, cols, conn_cols = [], [], []
for col in combined_conn.columns:
    if col != 'participant_id':
        m = re.match(pattern, col)
        if m:
            rows.append(int(m.group(1)))
            cols.append(int(m.group(2)))
            conn_cols.append(col)

num_samples, n_nodes = combined_conn.shape[0], 200
all_matrices = np.zeros((num_samples, n_nodes, n_nodes), dtype=np.float32)
values = combined_conn[conn_cols].values.astype(np.float32)

for idx, (r, c) in enumerate(zip(rows, cols)):
    all_matrices[:, r, c] = values[:, idx]
for i in range(num_samples):
    all_matrices[i] += all_matrices[i].T

conn_strength_df = pd.DataFrame(all_matrices.sum(axis=1), columns=[f"conn_strength_{i}" for i in range(n_nodes)])
conn_mean_df = pd.DataFrame(all_matrices.mean(axis=1), columns=[f"conn_mean_{i}" for i in range(n_nodes)])
conn_std_df = pd.DataFrame(all_matrices.std(axis=1), columns=[f"conn_std_{i}" for i in range(n_nodes)])

# Encode categorical features
le = LabelEncoder()
combined_cat["Ethnicity_encoded"] = le.fit_transform(combined_cat["PreInt_Demos_Fam_Child_Ethnicity"].astype(str))
combined_cat["Race_encoded"] = le.fit_transform(combined_cat["PreInt_Demos_Fam_Child_Race"].astype(str))
drop_cols = combined_cat.select_dtypes(include=['object', 'category']).columns.tolist() + ['participant_id', "Basic_Demos_Enroll_Year"]
combined_cat.drop(columns=drop_cols, errors="ignore", inplace=True)
combined_num.drop(columns=["participant_id"], errors="ignore", inplace=True)

# Final feature table
base_features = pd.concat([df.reset_index(drop=True) for df in [combined_num, combined_cat, conn_strength_df, conn_mean_df, conn_std_df]], axis=1)

n_train = train_num.shape[0]
train_features_full = base_features.iloc[:n_train, :].copy()
test_features_full = base_features.iloc[n_train:, :].copy()

sex_labels = labels['Sex_F'].astype(int).values
adhd_labels = labels['ADHD_Outcome'].astype(int).values

# Imputation and Variance Threshold
print("Applying variance threshold and KNN Imputation...")
vt = VarianceThreshold(threshold=1e-6)
train_vt = vt.fit_transform(train_features_full)
test_vt = vt.transform(test_features_full)

imputer = KNNImputer(n_neighbors=5)
train_imputed = imputer.fit_transform(train_vt)
test_imputed = imputer.transform(test_vt)
print("Preprocessing complete.")


print("Shape of train_imputed:", train_imputed.shape) 
# ===================================================================
# 4. SEX PREDICTION MODEL
# ===================================================================
print("\n=== Training SEX model ===")
# Get the number of features safely
if train_imputed.ndim == 1:
    num_features = 1
else:
    num_features = train_imputed.shape[1]

k_sex = min(300, num_features)

skb_sex = SelectKBest(score_func=mutual_info_classif, k=k_sex)
train_sex = skb_sex.fit_transform(train_imputed, sex_labels)
test_sex = skb_sex.transform(test_imputed)

params_sex = dict(
    booster="dart", n_estimators=800, max_depth=2,
    learning_rate=0.02, subsample=0.8, colsample_bytree=0.7,
    min_child_weight=6, gamma=0.05, reg_lambda=1.0, reg_alpha=0.2,
    rate_drop=0.1, skip_drop=0.1, sample_type="weighted", normalize_type="forest",
    scale_pos_weight=compute_class_weight(sex_labels),
    random_state=SEED, tree_method="hist", n_jobs=-1
)

oof_sex, test_pred_sex = kfold_oof_xgb(train_sex, sex_labels, test_sex, params=params_sex)

t_sex = 0.5 * (youden_threshold(sex_labels, oof_sex) + f1_threshold(sex_labels, oof_sex))
print(f"Selected SEX threshold: {t_sex:.4f}")

# Add sex prediction as a meta-feature
train_with_meta = np.column_stack([train_imputed, oof_sex.reshape(-1, 1)])
test_with_meta = np.column_stack([test_imputed, test_pred_sex.reshape(-1, 1)])

# ===================================================================
# 5. ADHD PREDICTION MODEL (with FLAML)
# ===================================================================
print("\n=== Tuning ADHD model with FLAML ===")

# Feature selection for ADHD
n_features_adhd = 1 if train_with_meta.ndim == 1 else train_with_meta.shape[1]
k_adhd = min(450, n_features_adhd)
skb_adhd = SelectKBest(score_func=mutual_info_classif, k=k_adhd)
train_adhd = skb_adhd.fit_transform(train_with_meta, adhd_labels)
test_adhd = skb_adhd.transform(test_with_meta)

from flaml import AutoML
automl = AutoML()

automl.fit(
    X_train=train_adhd,
    y_train=adhd_labels,
    task="classification",
    metric="roc_auc",
    estimator_list=["xgboost"],
    time_budget=FLAML_TIME_BUDGET,  # e.g., 600
    n_splits=5,                     # FLAML's internal CV
    log_file_name="flaml_adhd_log.log",
    seed=SEED,
    early_stop=True,
)

print("\n--- FLAML Tuning Complete ---")
print(f"Best ROC AUC found by FLAML: {1.0 - automl.best_loss:.4f}")
print("Best Hyperparameters:")
print(automl.best_config)

# Train final ADHD model with robust CV using best params
best_params_adhd = automl.best_config.copy()
best_params_adhd.update({
    "scale_pos_weight": compute_class_weight(adhd_labels),
    "random_state": SEED,
    "tree_method": "hist",
    "n_jobs": -1,
})

print("\n=== Training FINAL ADHD model with robust CV ===")
oof_adhd, test_pred_adhd = kfold_oof_xgb(
    train_adhd, adhd_labels, test_adhd,
    params=best_params_adhd,
    n_splits=N_SPLITS,
    early_stopping_rounds=70
)

# Threshold selection for ADHD
thr_adhd_youden = youden_threshold(adhd_labels, oof_adhd)
thr_adhd_f1 = f1_threshold(adhd_labels, oof_adhd)
t_adhd = float(np.clip(0.5 * (thr_adhd_youden + thr_adhd_f1), 0.05, 0.95))
print(f"Selected ADHD threshold: {t_adhd:.4f} (Youden={thr_adhd_youden:.4f}, F1={thr_adhd_f1:.4f})")


# ===================================================================
# 6. SUBMISSION
# ===================================================================
print("\nCreating submission file...")
submission = pd.DataFrame({
    "participant_id": test_cat['participant_id'].values,
    "ADHD_Outcome": (test_pred_adhd >= t_adhd).astype(int),
    "Sex_F": (test_pred_sex >= t_sex).astype(int),
})

OUT_PATH = Path("FLAML_XGB_Robust_Submission.csv")
submission.to_csv(OUT_PATH, index=False, sep=';')

print(f"\n✅ Submission saved to: {OUT_PATH.resolve()}")
print(f"   Columns order -> {list(submission.columns)}")